# 05 — Seaborn for signal analysis

**Read this first, because it will save you time:** seaborn is the least essential library in
this stack for a Signals and Systems course, and knowing *why* is more useful than memorising
its API.

Seaborn is built for **statistical relationships in tabular data**. A waveform is not that. If
you plot `sns.lineplot(x=t, y=x)` on a 10,000-sample signal, you get a slower, uglier version
of `ax.plot(t, x)` — seaborn will try to compute confidence intervals over your samples, which
is meaningless.

So where does it earn its place? Three situations, all of which you *will* hit:

1. **Comparing many experiments.** Forty filter designs across five families and six trials
   each. `sns.catplot` renders that comparison in one line; matplotlib needs thirty.
2. **Distributions.** Is this noise Gaussian? How does the residual distribution change after
   filtering? `histplot`, `kdeplot`, `ecdfplot`.
3. **Repeated trials with uncertainty.** Twenty runs of the same measurement, plotted as a
   mean with a confidence band — seaborn does the aggregation for you.

**The rule: matplotlib for signals, seaborn for the table of results about those signals.**

Everything here builds on notebook 04 — seaborn wants tidy (long-format) DataFrames, which is
exactly what `melt` and `groupby` produce.

---

## Contents

| § | Topic |
|---|-------|
| 1 | Setup, themes, and the matplotlib relationship |
| 2 | Figure-level vs. axes-level: the one concept that confuses everyone |
| 3 | Distributions: is my noise what I think it is? |
| 4 | Repeated trials and confidence bands |
| 5 | Categorical comparisons: the filter sweep |
| 6 | Faceting: many panels from one call |
| 7 | Heatmaps and correlation structure |
| 8 | Regression and trend |
| 9 | Pair plots for multi-channel data |
| 10 | Colour palettes done properly |
| 11 | When *not* to use seaborn |
| 12 | Exercises |

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import signal, stats

print("seaborn", sns.__version__)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
DATA = "../data"
rng = np.random.default_rng(11)

---
## 1. Setup, themes, and the matplotlib relationship

Seaborn *is* matplotlib underneath. Every seaborn function returns matplotlib objects, and
every matplotlib method still works on them. This is not a competing library; it is a
higher-level interface with better defaults.

In [ ]:
# sns.set_theme() changes matplotlib's rcParams globally -- your plain matplotlib plots
# get the seaborn look too.
fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))

with sns.axes_style("white"):
    pass  # (context managers let you scope a style)

t = np.linspace(0, 1, 500)
ax[0].plot(t, np.sin(2 * np.pi * 3 * t))
ax[0].set_title("plain matplotlib, seaborn theme active")

sns.lineplot(x=t, y=np.sin(2 * np.pi * 3 * t), ax=ax[1])
ax[1].set_title("sns.lineplot -- same result, more work")
fig.tight_layout()

In [ ]:
# The five built-in styles.
fig = plt.figure(figsize=(13, 2.4))
for i, style in enumerate(["darkgrid", "whitegrid", "dark", "white", "ticks"], start=1):
    with sns.axes_style(style):
        ax = fig.add_subplot(1, 5, i)
        ax.plot(t, np.sin(2 * np.pi * 3 * t))
        ax.set_title(style, fontsize=10)
fig.tight_layout()

> **Context, not just style.** `sns.set_theme(context=...)` scales all the fonts and line
> widths at once: `"paper"` < `"notebook"` < `"talk"` < `"poster"`. Set `context="talk"`
> before generating figures for a presentation and everything becomes legible from the back
> of the room without editing a single `fontsize`.

In [ ]:
fig = plt.figure(figsize=(13, 2.6))
for i, ctx in enumerate(["paper", "notebook", "talk"], start=1):
    with sns.plotting_context(ctx):
        ax = fig.add_subplot(1, 3, i)
        ax.plot(t, np.sin(2 * np.pi * 3 * t))
        ax.set_xlabel("time [s]"); ax.set_ylabel("amplitude")
        ax.set_title(f"context = {ctx}")
fig.tight_layout()

---
## 2. Figure-level vs. axes-level: the one concept that confuses everyone

Seaborn has two kinds of function and mixing them up is the source of most seaborn
frustration.

| | **Axes-level** | **Figure-level** |
|---|---|---|
| examples | `lineplot`, `scatterplot`, `histplot`, `boxplot`, `heatmap` | `relplot`, `displot`, `catplot`, `lmplot`, `pairplot` |
| draws onto | an existing Axes (`ax=` argument) | a whole new Figure it creates itself |
| returns | `matplotlib.axes.Axes` | a seaborn `FacetGrid` |
| can facet? | no | **yes** — `col=`, `row=` |
| use inside `plt.subplots`? | yes | **no** |

**Practical rule:** use axes-level functions when you are building a multi-panel figure
yourself. Use figure-level functions when you want seaborn to build the panels for you via
faceting. If you find yourself wanting to pass `ax=` to `relplot`, you wanted `lineplot`.

In [ ]:
# Axes-level: fits into a figure you control.
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
d = pd.DataFrame({"x": rng.normal(size=300), "y": rng.normal(size=300)})
sns.histplot(data=d, x="x", ax=ax[0])
sns.scatterplot(data=d, x="x", y="y", ax=ax[1], s=15)
ax[0].set_title("histplot (axes-level)"); ax[1].set_title("scatterplot (axes-level)")
fig.suptitle("Axes-level functions accept ax= and compose", y=1.03)
fig.tight_layout()

In [ ]:
# Figure-level: makes its own figure, and can facet.
d2 = d.assign(group=rng.choice(["A", "B", "C"], size=300))
g = sns.displot(data=d2, x="x", col="group", height=2.6, aspect=1.1)
g.figure.suptitle("displot (figure-level) -- three panels from one call", y=1.06)
print("returned object:", type(g).__name__)

---
## 3. Distributions: is my noise what I think it is?

The most genuinely useful seaborn application in this course. "Assume additive white Gaussian
noise" is an assumption you can and should check.

In [ ]:
n = 4000
noise = pd.DataFrame({
    "gaussian": rng.normal(0, 1, n),
    "uniform": rng.uniform(-1.7, 1.7, n),
    "laplace": rng.laplace(0, 0.7, n),
    "bimodal": np.concatenate([rng.normal(-1.5, 0.4, n // 2),
                               rng.normal(1.5, 0.4, n // 2)]),
}).melt(var_name="distribution", value_name="amplitude")

g = sns.displot(data=noise, x="amplitude", col="distribution",
                kde=True, height=2.6, aspect=1.0, bins=50, col_wrap=4)
g.figure.suptitle("Four noise sources with similar variance, very different character", y=1.06)

> **All four have comparable standard deviation.** Reporting only $\sigma$ would make them
> look interchangeable. They are not: the Laplace tails produce occasional large excursions
> that will trip a threshold detector, and the bimodal one is not noise at all — it is a
> two-state signal someone mislabelled.

In [ ]:
# ECDF is the honest way to compare distributions -- no bin-width choice to argue about.
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
sns.kdeplot(data=noise, x="amplitude", hue="distribution", ax=ax[0], common_norm=False)
sns.ecdfplot(data=noise, x="amplitude", hue="distribution", ax=ax[1])
ax[0].set_title("KDE -- smooth, but the bandwidth is a choice you made")
ax[1].set_title("ECDF -- no free parameters, nothing hidden")
fig.tight_layout()

> **Prefer the ECDF when you are making a claim.** A KDE's appearance depends on a bandwidth
> parameter, so two people can plot the same data and disagree about whether it is bimodal.
> The empirical CDF has no such knob.

In [ ]:
# Q-Q plot: the standard test for "is this Gaussian?".
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for a, col in zip(ax, ["gaussian", "uniform", "laplace", "bimodal"]):
    vals = noise.loc[noise["distribution"] == col, "amplitude"]
    stats.probplot(vals, dist="norm", plot=a)
    a.set_title(col)
    a.get_lines()[0].set_markersize(2)
fig.suptitle("Q-Q vs. normal: straight line = Gaussian", y=1.04)
fig.tight_layout()

In [ ]:
# The practical version: check the residual of a real filtering operation.
fs = 500.0
t = np.arange(2000) / fs
clean = np.sin(2 * np.pi * 8 * t)
observed = clean + rng.normal(0, 0.3, t.size)

sos = signal.butter(6, 25, fs=fs, output="sos")
filtered = signal.sosfiltfilt(sos, observed)

residuals = pd.DataFrame({
    "before filtering": observed - clean,
    "after filtering": filtered - clean,
}).melt(var_name="stage", value_name="residual")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
sns.histplot(data=residuals, x="residual", hue="stage", ax=ax[0],
             stat="density", element="step", common_norm=False, bins=60)
sns.ecdfplot(data=residuals, x="residual", hue="stage", ax=ax[1])
ax[0].set_title("Residual distributions"); ax[1].set_title("as ECDFs")
fig.tight_layout()

print(residuals.groupby("stage")["residual"].agg(["std", "min", "max"]).round(4))

---
## 4. Repeated trials and confidence bands

This is seaborn's genuine superpower for lab work. Give it many trials in long format and it
computes the mean and a bootstrapped confidence interval automatically.

In [ ]:
# 30 trials of the same measurement, at three noise levels.
fs, n_samples = 200.0, 400
t = np.arange(n_samples) / fs
records = []
for noise_level in [0.1, 0.3, 0.6]:
    for trial in range(30):
        y = np.exp(-2 * t) * np.sin(2 * np.pi * 6 * t) + rng.normal(0, noise_level, n_samples)
        records.append(pd.DataFrame({
            "time_s": t, "amplitude": y,
            "noise_sigma": noise_level, "trial": trial,
        }))
trials = pd.concat(records, ignore_index=True)
print(trials.head())
print(f"\n{len(trials)} rows = 3 noise levels x 30 trials x {n_samples} samples")

In [ ]:
# One call: mean across trials, with a 95% bootstrap CI band, per noise level.
g = sns.relplot(data=trials, x="time_s", y="amplitude", hue="noise_sigma",
                kind="line", errorbar=("ci", 95), height=3.4, aspect=2.4,
                palette="viridis_r")
g.set_axis_labels("time [s]", "amplitude")
g.figure.suptitle("Mean of 30 trials with 95% confidence band", y=1.04)

> **This is the one case where `sns.lineplot` on a time axis is correct.** There genuinely are
> repeated observations at each time point, so the aggregation is meaningful. On a single
> waveform there is one observation per time point and seaborn's aggregation machinery is
> pure overhead — that is when you go back to `ax.plot`.
>
> **`errorbar=` options:** `("ci", 95)` bootstrap CI (default, slow on big data), `"sd"`
> standard deviation, `("se", 2)` two standard errors, `("pi", 90)` percentile interval, or
> `None` to skip the aggregation entirely.

In [ ]:
# errorbar="sd" is much faster and often what you actually mean.
fig, ax = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for a, eb in zip(ax, [("ci", 95), "sd", ("pi", 90)]):
    sub = trials[trials["noise_sigma"] == 0.3]
    sns.lineplot(data=sub, x="time_s", y="amplitude", errorbar=eb, ax=a)
    a.set_title(f"errorbar = {eb}")
fig.tight_layout()

In [ ]:
# units= draws every individual trial faintly -- often more honest than a band.
fig, ax = plt.subplots(figsize=(10, 3.2))
sub = trials[trials["noise_sigma"] == 0.3]
sns.lineplot(data=sub, x="time_s", y="amplitude", units="trial", estimator=None,
             ax=ax, lw=0.4, alpha=0.25, color="tab:blue")
sns.lineplot(data=sub, x="time_s", y="amplitude", errorbar=None, ax=ax,
             lw=2.2, color="black")
ax.set(xlabel="time [s]", ylabel="amplitude",
       title="All 30 trials (faint) with the mean overlaid")

---
## 5. Categorical comparisons: the filter sweep

Back to `data/filter_sweep.csv` from notebook 04 — 750 measurements across five filter
families. This is the shape of data seaborn was designed for.

In [ ]:
sweep = pd.read_csv(f"{DATA}/filter_sweep.csv")
print(sweep.head())
print(f"\n{len(sweep)} rows, {sweep['family'].nunique()} families, "
      f"{sweep['order'].nunique()} orders, {sweep['cutoff_hz'].nunique()} cutoffs")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 6.5))

sns.boxplot(data=sweep, x="family", y="output_snr_db", ax=ax[0, 0])
ax[0, 0].set_title("boxplot -- quartiles and outliers")

sns.violinplot(data=sweep, x="family", y="output_snr_db", ax=ax[0, 1], inner="quart")
ax[0, 1].set_title("violinplot -- the full shape of the distribution")

sns.stripplot(data=sweep, x="family", y="output_snr_db", ax=ax[1, 0], size=2.5, alpha=0.5)
ax[1, 0].set_title("stripplot -- every single data point")

sns.barplot(data=sweep, x="family", y="output_snr_db", ax=ax[1, 1], errorbar="sd")
ax[1, 1].set_title("barplot -- mean ± sd (hides the distribution)")

fig.tight_layout()

> **The bar chart is the weakest of the four** and the one most often used. It shows two
> numbers per group and hides everything else — bimodality, outliers, sample size. Prefer
> `boxplot` when you have enough data, and overlay a `stripplot` when you do not.

In [ ]:
# Box + strip together: distribution summary and raw data in one panel.
fig, ax = plt.subplots(figsize=(10, 3.6))
sns.boxplot(data=sweep, x="family", y="output_snr_db", ax=ax,
            showfliers=False, boxprops=dict(alpha=0.5))
sns.stripplot(data=sweep, x="family", y="output_snr_db", ax=ax,
              size=2.5, alpha=0.4, color="0.2", jitter=0.25)
ax.set(xlabel="filter family", ylabel="output SNR [dB]",
       title="Box for the summary, points for the honesty")

In [ ]:
# hue splits each category further -- here, by filter order.
fig, ax = plt.subplots(figsize=(11, 3.6))
sns.boxplot(data=sweep, x="family", y="output_snr_db", hue="order", ax=ax, showfliers=False)
ax.set(xlabel="filter family", ylabel="output SNR [dB]",
       title="SNR by family and order")
ax.legend(title="order", bbox_to_anchor=(1.01, 1), loc="upper left")

In [ ]:
# pointplot is the clearest way to show an interaction (does order affect families equally?).
fig, ax = plt.subplots(figsize=(10, 3.6))
sns.pointplot(data=sweep, x="order", y="output_snr_db", hue="family",
              errorbar="se", dodge=0.4, ax=ax, markers="o", linestyles="-")
ax.set(xlabel="filter order", ylabel="output SNR [dB]",
       title="Non-parallel lines = the families respond differently to order")

---
## 6. Faceting: many panels from one call

`col=` and `row=` split the data into a grid of panels sharing axes. This is the argument
that makes figure-level functions worth the trouble.

In [ ]:
g = sns.catplot(data=sweep, x="order", y="output_snr_db", col="family",
                kind="box", height=2.8, aspect=0.85, col_wrap=5, showfliers=False)
g.set_axis_labels("order", "output SNR [dB]")
g.figure.suptitle("One panel per family, shared y-axis for honest comparison", y=1.05)

In [ ]:
# A 2-D facet grid: cutoff across columns, metric across rows.
long = sweep.melt(
    id_vars=["family", "order", "cutoff_hz", "trial"],
    value_vars=["output_snr_db", "group_delay_ms", "passband_ripple_db"],
    var_name="metric", value_name="value",
)

g = sns.relplot(data=long, x="order", y="value", hue="family",
                col="metric", kind="line", errorbar="sd",
                facet_kws=dict(sharey=False), height=3.0, aspect=1.15)
g.set_titles("{col_name}")
g.figure.suptitle("Three metrics vs. order -- note sharey=False, the scales differ", y=1.06)

> **`sharey=False` is a deliberate choice here** because the three metrics have different
> units. Whenever you turn sharing off, say so — an unwary reader will compare panel heights
> that are not comparable.

In [ ]:
# The engineering trade-off, as a scatter: SNR against delay, sized by ripple.
fig, ax = plt.subplots(figsize=(10, 4))
sns.scatterplot(data=sweep, x="group_delay_ms", y="output_snr_db",
                hue="family", size="passband_ripple_db", sizes=(10, 120),
                alpha=0.7, ax=ax)
ax.set(xlabel="group delay [ms]", ylabel="output SNR [dB]",
       title="Up and to the left is better; point size is passband ripple")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

---
## 7. Heatmaps and correlation structure

`sns.heatmap` wants a **wide** DataFrame — rows and columns already laid out. That is what
`pivot_table` produces.

In [ ]:
grid = sweep.pivot_table(index="order", columns="cutoff_hz",
                         values="output_snr_db", aggfunc="mean")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
sns.heatmap(grid, annot=True, fmt=".1f", cmap="viridis", ax=ax[0],
            cbar_kws={"label": "SNR [dB]"})
ax[0].set_title("sequential colormap -- for magnitudes")

centred = grid - grid.to_numpy().mean()
sns.heatmap(centred, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax[1],
            cbar_kws={"label": "SNR − mean [dB]"})
ax[1].set_title("diverging colormap -- only when zero means something")
fig.tight_layout()

> **`center=0` with a diverging colormap** is the rule for deviations, differences, and
> correlations. Using a diverging map without a meaningful midpoint invents a boundary in the
> data that is not there.

In [ ]:
# Correlation matrix of the accelerometer channels.
acc = pd.read_csv(f"{DATA}/accelerometer.csv", parse_dates=["timestamp"]).set_index("timestamp")
acc = acc.interpolate(limit=5).dropna()
acc = acc.assign(magnitude=np.sqrt(acc.acc_x**2 + acc.acc_y**2 + acc.acc_z**2))

corr = acc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # hide the redundant upper triangle

fig, ax = plt.subplots(figsize=(5.5, 4))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            cbar_kws={"label": "Pearson r"})
ax.set_title("Channel correlations")

In [ ]:
# A spectrogram-style heatmap: PSD per 5-second window.
fs = 100.0
z = acc["acc_z"].to_numpy()
win_len = int(5 * fs)
rows = []
for i in range(0, len(z) - win_len, win_len):
    f, P = signal.welch(z[i:i + win_len], fs, nperseg=256)
    rows.append(pd.Series(10 * np.log10(P + 1e-12), index=np.round(f, 1),
                          name=round(i / fs, 1)))
psd_table = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(psd_table.T.iloc[:80], cmap="magma", ax=ax,
            cbar_kws={"label": "PSD [dB]"})
ax.invert_yaxis()
ax.set_xlabel("window start [s]"); ax.set_ylabel("frequency [Hz]")
ax.set_title("Welch PSD per 5-second window -- the burst is the bright column")

> **Compare that against the `pcolormesh` spectrogram in notebook 02.** For a genuine
> spectrogram, `pcolormesh` is better: it gives real numeric axes rather than categorical
> tick labels. `sns.heatmap` is the right choice when the axes really are categories — filter
> family, experiment ID, window index.

---
## 8. Regression and trend

`regplot` (axes-level) and `lmplot` (figure-level) fit and draw a model with a confidence
band. Useful for checking whether a relationship you assumed is actually there.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))

sns.regplot(data=sweep, x="order", y="group_delay_ms", ax=ax[0],
            scatter_kws=dict(s=8, alpha=0.3))
ax[0].set_title("linear fit")

sns.regplot(data=sweep, x="cutoff_hz", y="group_delay_ms", ax=ax[1],
            scatter_kws=dict(s=8, alpha=0.3), order=2)
ax[1].set_title("order=2 polynomial")

sns.residplot(data=sweep, x="cutoff_hz", y="group_delay_ms", ax=ax[2],
              scatter_kws=dict(s=8, alpha=0.3))
ax[2].set_title("residuals of the linear fit -- structure means wrong model")
fig.tight_layout()

> **The residual panel is the important one.** Group delay goes as $1/f_c$, so a straight-line
> fit leaves obvious curvature in the residuals. A residual plot that still has structure is
> telling you the model is wrong — a fact the $R^2$ of the original fit would happily hide.

In [ ]:
# lmplot facets the regression, which regplot cannot do.
g = sns.lmplot(data=sweep, x="order", y="output_snr_db", col="family",
               height=2.7, aspect=0.9, col_wrap=5,
               scatter_kws=dict(s=8, alpha=0.3), logx=False)
g.figure.suptitle("Per-family trend of SNR against order", y=1.05)

---
## 9. Pair plots for multi-channel data

`pairplot` gives every pairwise scatter plus the marginal distributions. For a handful of
channels it is a fast way to spot structure you were not looking for.

In [ ]:
sample = acc[["acc_x", "acc_y", "acc_z"]].iloc[::20]      # subsample -- pairplot is slow
g = sns.pairplot(sample, height=1.9, plot_kws=dict(s=6, alpha=0.25),
                 diag_kind="kde", corner=True)
g.figure.suptitle("Accelerometer channels, pairwise", y=1.02)

> **`corner=True`** drops the redundant upper triangle. **Subsample first** — `pairplot` on
> 12,000 points draws 9 panels of 12,000 markers each and will make your notebook crawl.

In [ ]:
# jointplot: one pair, in more detail, with marginals.
g = sns.jointplot(data=sample, x="acc_x", y="acc_z", kind="hex", height=4.5)
g.figure.suptitle("Hexbin joint distribution", y=1.02)

---
## 10. Colour palettes done properly

Seaborn's palette handling is genuinely better than raw matplotlib's, and the categories map
directly onto the kinds of data you have.

In [ ]:
palettes = {
    "deep (categorical)": sns.color_palette("deep", 8),
    "colorblind (categorical)": sns.color_palette("colorblind", 8),
    "viridis (sequential)": sns.color_palette("viridis", 8),
    "rocket (sequential)": sns.color_palette("rocket", 8),
    "vlag (diverging)": sns.color_palette("vlag", 8),
    "Set2 (categorical, soft)": sns.color_palette("Set2", 8),
}

fig, ax = plt.subplots(len(palettes), 1, figsize=(8, 5))
for a, (name, pal) in zip(ax, palettes.items()):
    for i, c in enumerate(pal):
        a.barh(0, 1, left=i, color=c, height=1)
    a.set_xlim(0, 8); a.set_yticks([]); a.set_xticks([])
    a.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
    a.grid(False)
fig.tight_layout()

| kind of data | palette | why |
|---|---|---|
| unordered categories (filter families) | `deep`, `Set2`, **`colorblind`** | distinct hues, no implied order |
| ordered categories (filter order 2→10) | `viridis`, `rocket`, `mako` | brightness encodes the order |
| deviations from a reference | `vlag`, `coolwarm`, `RdBu_r` | two directions from a neutral centre |

**Use `colorblind` by default for categorical data.** It costs nothing and roughly 1 in 12
men cannot reliably distinguish the `deep` palette's red and green.

In [ ]:
sns.set_palette("colorblind")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
sns.boxplot(data=sweep, x="family", y="output_snr_db", ax=ax[0], showfliers=False)
ax[0].set_title("categorical -> colorblind palette")

sns.lineplot(data=sweep, x="cutoff_hz", y="output_snr_db", hue="order",
             palette="viridis", errorbar="sd", ax=ax[1])
ax[1].set_title("ordered hue -> sequential palette")
ax[1].legend(title="order", fontsize=8)
fig.tight_layout()

---
## 11. When *not* to use seaborn

Worth stating explicitly, because the failure mode is silent — you get a plot, it just is not
the plot you wanted.

In [ ]:
import time

fs = 5000.0
t = np.arange(int(fs * 4)) / fs
x = signal.chirp(t, f0=10, f1=800, t1=4) + rng.normal(0, 0.1, t.size)
wave = pd.DataFrame({"t": t, "x": x})

start = time.perf_counter()
fig, ax = plt.subplots(figsize=(10, 2.2))
sns.lineplot(data=wave, x="t", y="x", ax=ax, errorbar=None)
ax.set_title("sns.lineplot on a 20,000-sample waveform")
t_sns = time.perf_counter() - start

start = time.perf_counter()
fig, ax = plt.subplots(figsize=(10, 2.2))
ax.plot(t, x, lw=0.4)
ax.set_title("ax.plot on the same data")
t_mpl = time.perf_counter() - start

print(f"seaborn    : {t_sns * 1000:7.1f} ms")
print(f"matplotlib : {t_mpl * 1000:7.1f} ms   ({t_sns / t_mpl:.1f}x faster)")

**Do not reach for seaborn when:**

| situation | use instead |
|---|---|
| plotting a single waveform | `ax.plot(t, x)` |
| a discrete-time sequence $x[n]$ | `ax.stem(n, x)` |
| a Bode plot | `ax.semilogx` — seaborn has no log-frequency idiom |
| a pole-zero map | matplotlib with `set_aspect("equal")` |
| a true spectrogram | `ax.pcolormesh` — real numeric axes, not categorical ticks |
| anything above ~50k points | matplotlib, or downsample first |

**Do reach for seaborn when:** you have a tidy DataFrame with a categorical column you want
to split by, repeated trials to aggregate, a distribution to characterise, or a facet grid to
build.

---
## 12. Exercises

**1. Noise characterisation.** Load `bench_capture.txt` (see notebook 04 for the loader),
high-pass it at 20 Hz to isolate the noise, and characterise the residual: histogram, ECDF,
Q-Q plot, and a formal normality test (`scipy.stats.shapiro` or `normaltest`). Is the noise
Gaussian? State your conclusion with the p-value.

**2. Trial aggregation from real data.** Split the ECG recording into individual beats aligned
on the R-peak, assemble them into long format, and use `sns.lineplot` to show the mean beat
shape with a confidence band. This plot is standard in every cardiology paper.

**3. Window function comparison.** For each of six window functions, measure main-lobe width
and peak sidelobe level across several signal lengths. Build a tidy DataFrame and use
`catplot` to show the trade-off. Which window would you pick for a spectrum analyser, and why?

**4. Facet the filter sweep differently.** Reproduce the §6 facet grid with `row="family"` and
`col="cutoff_hz"` instead. Which layout answers "does cutoff matter more than family?" more
directly? Argue for one.

**5. Colorblind audit.** Take three figures from this notebook, convert them to greyscale
(save as PNG and use PIL), and identify which remain readable. Fix the ones that do not.

**6. Interaction plot.** Using `pointplot`, determine whether filter order and cutoff interact
in their effect on SNR — that is, whether the effect of order depends on the cutoff. Support
your visual answer with a two-way ANOVA (`scipy.stats` or `statsmodels`).

**7. Build the plot seaborn cannot.** Make a figure with a Bode magnitude panel, a Bode phase
panel, and a pole-zero map, all styled consistently with `sns.set_theme()` but drawn entirely
in matplotlib. This is the realistic end state: seaborn for the theme and the statistics,
matplotlib for the signals.

**8. One figure, one story.** Take everything in `data/` and produce a *single* figure that
makes one clear claim about the accelerometer burst. Constraints: at most four panels, every
axis labelled with units, readable in greyscale, and no panel that does not support the claim.

---

### Quick reference

| Task | Call |
|------|------|
| set theme | `sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")` |
| scoped style | `with sns.axes_style("ticks"): ...` |
| distribution | `sns.histplot(data=df, x=, hue=, stat="density", element="step")` |
| smooth density | `sns.kdeplot(data=df, x=, hue=, common_norm=False)` |
| cumulative | `sns.ecdfplot(data=df, x=, hue=)` |
| trials + CI | `sns.lineplot(data=df, x=, y=, errorbar=("ci", 95))` |
| individual trials | `sns.lineplot(..., units="trial", estimator=None)` |
| category compare | `sns.boxplot`, `sns.violinplot`, `sns.stripplot`, `sns.pointplot` |
| facet grid | `sns.catplot(..., col=, row=, col_wrap=)` / `sns.relplot(...)` |
| matrix | `sns.heatmap(wide_df, annot=True, cmap=, center=0)` |
| trend + CI | `sns.regplot(data=, x=, y=, order=2)` |
| model check | `sns.residplot(data=, x=, y=)` |
| all pairs | `sns.pairplot(df, corner=True, diag_kind="kde")` |
| unshared facet axes | `facet_kws=dict(sharey=False)` |

### The three things to remember

1. **Tidy (long) data in.** If seaborn is being awkward, the answer is almost always `melt`.
2. **Axes-level takes `ax=`; figure-level takes `col=`/`row=`.** Pick based on who builds the figure.
3. **Seaborn for tables about signals, matplotlib for the signals themselves.**